# LeetCode 1636 — Sort Array by Increasing Frequency bằng Python

Link đề: https://leetcode.com/problems/sort-array-by-increasing-frequency/

Độ khó: **Easy** · Chủ đề: Array, Hash Table, Sorting

Skeleton LeetCode:

```python
class Solution:
    def frequencySort(self, nums: list[int]) -> list[int]:

```

Mục tiêu học:

1. Hiểu đề: sắp xếp theo **tần suất tăng dần**, hòa tần suất thì theo **giá trị giảm dần**.
2. Biết đếm tần suất bằng `dict` tự viết và bằng `collections.Counter`.
3. Nắm chắc `sorted(..., key=...)`, `lambda`, và mẹo **so sánh tuple** + **đảo dấu** để sắp xếp nhiều tiêu chí.
4. Thấy được cách trực giác `nums.count(x)` chậm ở đâu, rồi cải tiến thành đếm một lần.
5. Biết thêm cách "bucket theo tần suất" và `functools.cmp_to_key` để so sánh kiểu comparator.
6. Tự viết lại lời giải và giải được các biến thể sắp xếp theo nhiều khóa.

## 1. Bài toán yêu cầu gì?

Cho mảng số nguyên `nums`. Hãy sắp xếp lại mảng sao cho:

- Số nào **xuất hiện ít lần hơn** đứng **trước**.
- Nếu hai số có **cùng số lần xuất hiện**, số **lớn hơn** đứng trước (giảm dần theo giá trị).

Trả về mảng đã sắp xếp (mỗi số vẫn giữ đủ số lần xuất hiện của nó).

| nums | Kết quả | Vì sao? |
|---|---|---|
| `[1,1,2,2,2,3]` | `[3,1,1,2,2,2]` | tần suất: 3→1, 1→2, 2→3 |
| `[2,3,1,3,2]` | `[1,3,3,2,2]` | 1→1 lần; 2 và 3 đều 2 lần nên 3 (lớn hơn) đứng trước 2 |
| `[-1,1,-6,4,5,-6,1,4,1]` | `[5,-1,4,4,-6,-6,1,1,1]` | 5,-1: 1 lần (5 > -1); 4,-6: 2 lần (4 > -6); 1: 3 lần |

Constraints:

- `1 <= nums.length <= 100`
- `-100 <= nums[i] <= 100`

Chú ý: có **số âm**, nên "giảm dần theo giá trị" nghĩa là `5` trước `-1`, `4` trước `-6`.

In [1]:
nums = [-1, 1, -6, 4, 5, -6, 1, 4, 1]
expected = [5, -1, 4, 4, -6, -6, 1, 1, 1]

print("Đầu vào :", nums)
print("Kỳ vọng :", expected)
print("Số phần tử giữ nguyên?", len(nums) == len(expected))
print("Cùng các phần tử (chỉ khác thứ tự)?", sorted(nums) == sorted(expected))

Đầu vào : [-1, 1, -6, 4, 5, -6, 1, 4, 1]
Kỳ vọng : [5, -1, 4, 4, -6, -6, 1, 1, 1]
Số phần tử giữ nguyên? True
Cùng các phần tử (chỉ khác thứ tự)? True


### Tự làm bằng tay trước khi code

Với `nums = [2,3,1,3,2]`:

1. Đếm: `1` xuất hiện 1 lần, `2` xuất hiện 2 lần, `3` xuất hiện 2 lần.
2. Xếp nhóm theo tần suất tăng dần: tần suất 1 → `{1}`; tần suất 2 → `{2, 3}`.
3. Trong nhóm tần suất 2, xếp giá trị giảm dần: `3` rồi `2`.
4. "Bung" mỗi số ra đủ số lần: `[1] + [3,3] + [2,2] = [1,3,3,2,2]`.

Hai việc chính: **đếm** và **sắp xếp theo 2 tiêu chí**. Notebook sẽ dạy từng việc.

## 2. Nền tảng Python cần biết

### 2.1. `dict` để đếm tần suất

`dict` (dictionary) lưu các cặp **key → value**. Ở bài này: key là số, value là số lần xuất hiện.

- `freq.get(x, 0)` đọc giá trị của key `x`, nếu chưa có thì trả `0`.
- `freq[x] = freq.get(x, 0) + 1` tăng bộ đếm của `x` thêm 1.

In [2]:
nums = [2, 3, 1, 3, 2]
freq = {}
for x in nums:
    freq[x] = freq.get(x, 0) + 1
    print(f"gặp {x} -> freq = {freq}")

print("Tần suất cuối:", freq)
print("Số 3 xuất hiện:", freq[3], "lần")

gặp 2 -> freq = {2: 1}
gặp 3 -> freq = {2: 1, 3: 1}
gặp 1 -> freq = {2: 1, 3: 1, 1: 1}
gặp 3 -> freq = {2: 1, 3: 2, 1: 1}
gặp 2 -> freq = {2: 2, 3: 2, 1: 1}
Tần suất cuối: {2: 2, 3: 2, 1: 1}
Số 3 xuất hiện: 2 lần


### 2.2. `collections.Counter` — đếm sẵn trong thư viện chuẩn

`Counter` là một loại `dict` chuyên đếm. `Counter(nums)` đếm hết trong một dòng.
Truy cập key không có trong Counter sẽ trả `0` (không báo lỗi như `dict` thường).

In [3]:
from collections import Counter

cnt = Counter([2, 3, 1, 3, 2])
print(cnt)
print("cnt[2] =", cnt[2])
print("cnt[99] =", cnt[99], "(không có thì trả 0)")
print("Có giống dict tự đếm không?", dict(cnt) == {2: 2, 3: 2, 1: 1})

Counter({2: 2, 3: 2, 1: 1})
cnt[2] = 2
cnt[99] = 0 (không có thì trả 0)
Có giống dict tự đếm không? True


### 2.3. `sorted()` và `list.sort()`

- `sorted(a)` trả về **list mới** đã sắp xếp, không đổi `a`.
- `a.sort()` sắp xếp **tại chỗ**, trả về `None`.
- Mặc định sắp xếp tăng dần; `reverse=True` để giảm dần.

In [4]:
a = [3, -1, 2]
b = sorted(a)
print("a =", a, "| b = sorted(a) =", b)

a.sort()
print("sau a.sort(), a =", a)
print("a.sort() trả về:", [3, 1].sort())
print("giảm dần:", sorted([3, -1, 2], reverse=True))

a = [3, -1, 2] | b = sorted(a) = [-1, 2, 3]
sau a.sort(), a = [-1, 2, 3]
a.sort() trả về: None
giảm dần: [3, 2, -1]


### 2.4. Tham số `key` và `lambda`

`key` là một **hàm** nhận một phần tử và trả về "nhãn" dùng để so sánh. Python sắp xếp theo nhãn, nhưng list kết quả vẫn chứa phần tử gốc.

`lambda x: ...` là cách viết hàm ngắn một dòng, không cần `def`.

Ví dụ: sắp xếp từ theo độ dài.

In [5]:
words = ["banana", "kiwi", "apple", "fig"]
print(sorted(words, key=lambda w: len(w)))

# lambda tương đương với hàm def:
def length_of(w):
    return len(w)
print(sorted(words, key=length_of))

['fig', 'kiwi', 'apple', 'banana']
['fig', 'kiwi', 'apple', 'banana']


### 2.5. So sánh tuple — chìa khóa để sắp xếp nhiều tiêu chí

Python so sánh tuple **từ trái sang phải**: so phần tử đầu; nếu bằng nhau mới so phần tử thứ hai, v.v.

```text
(1, 5) < (2, 0)   vì 1 < 2 (không cần nhìn phần sau)
(2, 3) < (2, 7)   vì phần đầu bằng nhau, so tiếp 3 < 7
```

Nên `key=lambda x: (tiêu_chí_1, tiêu_chí_2)` cho phép sắp xếp theo tiêu chí 1, hòa thì xét tiêu chí 2.

In [6]:
print((1, 5) < (2, 0))
print((2, 3) < (2, 7))
print((2, 7) < (2, 3))
print(sorted([(2, 7), (1, 5), (2, 3)]))

True
True
False
[(1, 5), (2, 3), (2, 7)]


### 2.6. Mẹo đảo dấu để "một tiêu chí tăng, một tiêu chí giảm"

Đề muốn: tần suất **tăng**, giá trị **giảm**. `sorted` chỉ sắp tăng theo key.

Mẹo: dùng `-x` cho tiêu chí muốn giảm. Vì `a > b` tương đương `-a < -b`, sắp tăng theo `-x` chính là sắp giảm theo `x`.

Với số âm cũng đúng: `5 > -1` nên `-5 < 1`, tức `5` đứng trước `-1`.

In [7]:
vals = [5, -1, 4, -6]
print("tăng theo x  :", sorted(vals))
print("tăng theo -x :", sorted(vals, key=lambda x: -x))
print("kiểm tra -5 < 1:", -5 < 1)

tăng theo x  : [-6, -1, 4, 5]
tăng theo -x : [5, 4, -1, -6]
kiểm tra -5 < 1: True


## 3. Cách trực giác — đếm bằng `nums.count(x)` ngay trong key

Người mới thường nghĩ: "Tần suất của `x` là `nums.count(x)`, vậy cứ đưa vào key".

```python
sorted(nums, key=lambda x: (nums.count(x), -x))
```

`nums.count(x)` duyệt cả mảng để đếm `x`, tốn `O(n)` mỗi lần gọi.

In [8]:
def frequency_sort_count(nums: list[int]) -> list[int]:
    return sorted(nums, key=lambda x: (nums.count(x), -x))

print(frequency_sort_count([1, 1, 2, 2, 2, 3]))
print(frequency_sort_count([2, 3, 1, 3, 2]))
print(frequency_sort_count([-1, 1, -6, 4, 5, -6, 1, 4, 1]))

[3, 1, 1, 2, 2, 2]
[1, 3, 3, 2, 2]
[5, -1, 4, 4, -6, -6, 1, 1, 1]


### Nút thắt của cách trực giác

`sorted` gọi hàm `key` **một lần cho mỗi phần tử** (n lần). Mỗi lần lại gọi `nums.count(x)` tốn `O(n)`.

Tổng: `O(n²)` chỉ để tính key, cộng `O(n log n)` để sắp xếp.

Ta đang **đếm đi đếm lại** cùng một số: nếu `1` xuất hiện 3 lần thì `nums.count(1)` bị gọi 3 lần, mỗi lần duyệt cả mảng.

Cell dưới đếm số phép so sánh mà `count` phải làm.

In [9]:
def count_work(nums):
    work = 0
    for x in nums:          # sorted gọi key cho từng phần tử
        work += len(nums)   # nums.count(x) duyệt toàn bộ mảng
    return work

for n in [10, 100, 1000]:
    print(f"n = {n:>4}: khoảng {count_work(list(range(n))):>8} bước đếm")

n =   10: khoảng      100 bước đếm
n =  100: khoảng    10000 bước đếm
n = 1000: khoảng  1000000 bước đếm


## 4. Insight — đếm một lần, tra cứu nhiều lần

Ý tưởng: **tách việc đếm ra trước**.

1. Đếm tần suất của tất cả các số **một lần** vào `freq` (dict/Counter): `O(n)`.
2. Khi sắp xếp, key của `x` chỉ là tra cứu `freq[x]`: trung bình `O(1)`.

Key cho mỗi phần tử:

```python
(freq[x], -x)
```

- Phần đầu `freq[x]`: tần suất nhỏ đứng trước.
- Phần sau `-x`: hòa tần suất thì giá trị lớn đứng trước.

Thử tính key bằng tay cho `[2,3,1,3,2]`:

| x | freq[x] | -x | key |
|---:|---:|---:|---|
| 2 | 2 | -2 | (2, -2) |
| 3 | 2 | -3 | (2, -3) |
| 1 | 1 | -1 | (1, -1) |

Sắp tăng các key: `(1,-1)` < `(2,-3)` < `(2,-2)` → `1, 3, 3, 2, 2`.

In [10]:
from collections import Counter

nums = [2, 3, 1, 3, 2]
freq = Counter(nums)
for x in nums:
    print(f"x = {x:>2} -> key = {(freq[x], -x)}")

x =  2 -> key = (2, -2)
x =  3 -> key = (2, -3)
x =  1 -> key = (1, -1)
x =  3 -> key = (2, -3)
x =  2 -> key = (2, -2)


## 5. Lời giải chính — Counter + sorted với key tuple

Giảng từng đoạn:

**Đoạn 1 — đếm:**

```python
freq = Counter(nums)
```

**Đoạn 2 — định nghĩa key:** tần suất trước, giá trị đảo dấu sau.

```python
key = lambda x: (freq[x], -x)
```

**Đoạn 3 — sắp xếp và trả về:**

```python
return sorted(nums, key=key)
```

Ghép lại:

In [11]:
from collections import Counter

def frequency_sort_counter(nums: list[int]) -> list[int]:
    freq = Counter(nums)                       # đếm một lần
    return sorted(nums, key=lambda x: (freq[x], -x))

print(frequency_sort_counter([1, 1, 2, 2, 2, 3]))
print(frequency_sort_counter([2, 3, 1, 3, 2]))
print(frequency_sort_counter([-1, 1, -6, 4, 5, -6, 1, 4, 1]))

[3, 1, 1, 2, 2, 2]
[1, 3, 3, 2, 2]
[5, -1, 4, 4, -6, -6, 1, 1, 1]


### Phiên bản không dùng Counter (tự đếm bằng dict)

Nếu phỏng vấn yêu cầu không dùng thư viện, ta tự đếm như mục 2.1. Logic sắp xếp giữ nguyên.

In [12]:
def frequency_sort_dict(nums: list[int]) -> list[int]:
    freq = {}
    for x in nums:
        freq[x] = freq.get(x, 0) + 1
    return sorted(nums, key=lambda x: (freq[x], -x))

print(frequency_sort_dict([2, 3, 1, 3, 2]))

[1, 3, 3, 2, 2]


## 6. TRACE từng bước

Ta in ra: bảng tần suất, key của từng phần tử **trước khi sắp**, rồi danh sách cặp `(key, x)` **sau khi sắp**.

Bảng trace cho `nums = [-1,1,-6,4,5,-6,1,4,1]`:

| x | freq[x] | key = (freq, -x) |
|---:|---:|---|
| 5 | 1 | (1, -5) |
| -1 | 1 | (1, 1) |
| 4 | 2 | (2, -4) |
| -6 | 2 | (2, 6) |
| 1 | 3 | (3, -1) |

Xếp key tăng dần: `(1,-5) < (1,1) < (2,-4) < (2,6) < (3,-1)` → `5, -1, 4, 4, -6, -6, 1, 1, 1`.

In [13]:
from collections import Counter

def trace_frequency_sort(nums: list[int]) -> list[int]:
    freq = Counter(nums)
    print("Bước 1 — đếm tần suất:", dict(freq))
    print("Bước 2 — key của từng phần tử (theo thứ tự gốc):")
    for i, x in enumerate(nums):
        print(f"   i={i}, x={x:>3}, key=({freq[x]}, {-x:>3})")
    result = sorted(nums, key=lambda x: (freq[x], -x))
    print("Bước 3 — sau khi sắp theo key:")
    for x in result:
        print(f"   key=({freq[x]}, {-x:>3}) -> {x}")
    print("Kết quả:", result)
    return result

trace_frequency_sort([-1, 1, -6, 4, 5, -6, 1, 4, 1])

Bước 1 — đếm tần suất: {-1: 1, 1: 3, -6: 2, 4: 2, 5: 1}
Bước 2 — key của từng phần tử (theo thứ tự gốc):
   i=0, x= -1, key=(1,   1)
   i=1, x=  1, key=(3,  -1)
   i=2, x= -6, key=(2,   6)
   i=3, x=  4, key=(2,  -4)
   i=4, x=  5, key=(1,  -5)
   i=5, x= -6, key=(2,   6)
   i=6, x=  1, key=(3,  -1)
   i=7, x=  4, key=(2,  -4)
   i=8, x=  1, key=(3,  -1)
Bước 3 — sau khi sắp theo key:
   key=(1,  -5) -> 5
   key=(1,   1) -> -1
   key=(2,  -4) -> 4
   key=(2,  -4) -> 4
   key=(2,   6) -> -6
   key=(2,   6) -> -6
   key=(3,  -1) -> 1
   key=(3,  -1) -> 1
   key=(3,  -1) -> 1
Kết quả: [5, -1, 4, 4, -6, -6, 1, 1, 1]


[5, -1, 4, 4, -6, -6, 1, 1, 1]

### `enumerate` là gì?

Ở cell trace có `for i, x in enumerate(nums)`: mỗi vòng cho cả **chỉ số** `i` và **giá trị** `x`.

In [14]:
for i, x in enumerate(["a", "b", "c"]):
    print(i, x)

0 a
1 b
2 c


## 7. Code nộp LeetCode

Chữ ký đúng là `frequencySort(self, nums: list[int]) -> list[int]` trong `class Solution`.

In [15]:
from collections import Counter

class Solution:
    def frequencySort(self, nums: list[int]) -> list[int]:
        freq = Counter(nums)
        return sorted(nums, key=lambda x: (freq[x], -x))

sol = Solution()
assert sol.frequencySort([1, 1, 2, 2, 2, 3]) == [3, 1, 1, 2, 2, 2]
assert sol.frequencySort([2, 3, 1, 3, 2]) == [1, 3, 3, 2, 2]
assert sol.frequencySort([-1, 1, -6, 4, 5, -6, 1, 4, 1]) == [5, -1, 4, 4, -6, -6, 1, 1, 1]
print("3 ví dụ của đề PASS")

3 ví dụ của đề PASS


## 8. Các lời giải khác để mở rộng tư duy

### 8.1. Cách bucket theo tần suất (không cần sort trên toàn mảng)

Quan sát: tần suất nằm trong `1..n`, giá trị nằm trong `-100..100`. Ta có thể:

1. Đếm `freq`.
2. Tạo `buckets[f]` = danh sách các **giá trị khác nhau** có tần suất `f`.
3. Duyệt `f` từ `1` đến `n`; trong mỗi bucket, xếp giá trị giảm dần; mỗi giá trị thêm vào kết quả `f` lần.

`[x] * f` tạo list gồm `f` bản sao của `x`, ví dụ `[3] * 2 == [3, 3]`.

In [16]:
print([3] * 2)
print([7] * 0)

buckets = [[] for _ in range(4)]
print("4 bucket rỗng độc lập:", buckets)

[3, 3]
[]
4 bucket rỗng độc lập: [[], [], [], []]


In [17]:
from collections import Counter

def frequency_sort_bucket(nums: list[int]) -> list[int]:
    freq = Counter(nums)
    n = len(nums)
    buckets = [[] for _ in range(n + 1)]   # buckets[f] chứa các giá trị có tần suất f
    for value, f in freq.items():
        buckets[f].append(value)
    result = []
    for f in range(1, n + 1):
        for value in sorted(buckets[f], reverse=True):
            result.extend([value] * f)
    return result

print(frequency_sort_bucket([1, 1, 2, 2, 2, 3]))
print(frequency_sort_bucket([2, 3, 1, 3, 2]))
print(frequency_sort_bucket([-1, 1, -6, 4, 5, -6, 1, 4, 1]))

[3, 1, 1, 2, 2, 2]
[1, 3, 3, 2, 2]
[5, -1, 4, 4, -6, -6, 1, 1, 1]


Giải thích `freq.items()`: trả về các cặp `(key, value)`, ở đây là `(giá_trị, tần_suất)`.

`result.extend(lst)` nối tất cả phần tử của `lst` vào cuối `result` (khác `append` thêm cả list như một phần tử).

In [18]:
r = [1]
r.extend([3, 3])
print("extend:", r)
r2 = [1]
r2.append([3, 3])
print("append:", r2)

extend: [1, 3, 3]
append: [1, [3, 3]]


Code nộp LeetCode — cách bucket:

In [19]:
from collections import Counter

class SolutionBucket:
    def frequencySort(self, nums: list[int]) -> list[int]:
        freq = Counter(nums)
        buckets = [[] for _ in range(len(nums) + 1)]
        for value, f in freq.items():
            buckets[f].append(value)
        result = []
        for f in range(1, len(nums) + 1):
            for value in sorted(buckets[f], reverse=True):
                result.extend([value] * f)
        return result

assert SolutionBucket().frequencySort([2, 3, 1, 3, 2]) == [1, 3, 3, 2, 2]
print("SolutionBucket PASS")

SolutionBucket PASS


### 8.2. Cách comparator với `functools.cmp_to_key`

Hint của đề nói "custom comparator". Comparator là hàm `cmp(a, b)` trả về:

- số **âm** nếu `a` phải đứng trước `b`,
- số **dương** nếu `a` phải đứng sau `b`,
- `0` nếu coi như bằng nhau.

Python 3 không nhận comparator trực tiếp, phải bọc bằng `cmp_to_key`.

In [20]:
from collections import Counter
from functools import cmp_to_key

def frequency_sort_cmp(nums: list[int]) -> list[int]:
    freq = Counter(nums)

    def cmp(a: int, b: int) -> int:
        if freq[a] != freq[b]:
            return freq[a] - freq[b]   # tần suất nhỏ đứng trước
        return b - a                   # hòa: giá trị lớn đứng trước

    return sorted(nums, key=cmp_to_key(cmp))

print(frequency_sort_cmp([2, 3, 1, 3, 2]))
print(frequency_sort_cmp([-1, 1, -6, 4, 5, -6, 1, 4, 1]))

[1, 3, 3, 2, 2]
[5, -1, 4, 4, -6, -6, 1, 1, 1]


### 8.3. Hai lần sort ổn định (stable sort)

`sorted` của Python là **ổn định**: phần tử có key bằng nhau giữ nguyên thứ tự tương đối trước khi sắp.

Mẹo: sắp theo **tiêu chí phụ trước**, rồi sắp theo **tiêu chí chính sau**.

1. Sắp giảm theo giá trị.
2. Sắp tăng theo tần suất — các số cùng tần suất vẫn giữ thứ tự giảm từ bước 1.

In [21]:
from collections import Counter

def frequency_sort_two_pass(nums: list[int]) -> list[int]:
    freq = Counter(nums)
    step1 = sorted(nums, reverse=True)          # tiêu chí phụ
    step2 = sorted(step1, key=lambda x: freq[x])  # tiêu chí chính
    return step2

print(frequency_sort_two_pass([2, 3, 1, 3, 2]))
print(frequency_sort_two_pass([-1, 1, -6, 4, 5, -6, 1, 4, 1]))

[1, 3, 3, 2, 2]
[5, -1, 4, 4, -6, -6, 1, 1, 1]


### 8.4. Bảng so sánh các cách

| Cách | Ý tưởng | Thời gian | Bộ nhớ phụ | Nhận xét |
|---|---|---|---|---|
| `nums.count` trong key | đếm lại mỗi phần tử | `O(n²)` | `O(n)` | dễ viết, chậm khi n lớn |
| Counter + key tuple | đếm 1 lần, key `(freq, -x)` | `O(n log n)` | `O(n)` | **nên dùng** |
| Bucket theo tần suất | nhóm theo f, xếp trong nhóm | `O(n + k log k)` (k = số giá trị khác nhau) | `O(n)` | hay khi nhiều phần tử trùng |
| Comparator `cmp_to_key` | so sánh từng cặp | `O(n log n)` | `O(n)` | giống tư duy Java/C++ |
| Hai lần stable sort | phụ trước, chính sau | `O(n log n)` | `O(n)` | minh họa tính ổn định |

## 9. Vì sao lời giải đúng?

Gọi `key(x) = (freq[x], -x)`.

- Với hai số `a`, `b` khác tần suất: tuple so phần đầu, nên số có tần suất nhỏ hơn đứng trước. Đúng yêu cầu 1.
- Với hai số cùng tần suất: phần đầu bằng nhau, so `-a` với `-b`. `-a < -b` ⇔ `a > b`, nên số lớn hơn đứng trước. Đúng yêu cầu 2.
- Hai phần tử **cùng giá trị** có cùng key, nên chúng nằm cạnh nhau thành một khối liền.
- `sorted` chỉ đổi thứ tự, không thêm/bớt phần tử, nên mỗi giá trị vẫn giữ đủ số lần.

Invariant của mảng kết quả: đọc từ trái sang phải, key **không giảm**.

In [22]:
from collections import Counter

def check_order(result):
    freq = Counter(result)
    keys = [(freq[x], -x) for x in result]
    return all(keys[i] <= keys[i + 1] for i in range(len(keys) - 1))

r = frequency_sort_counter([-1, 1, -6, 4, 5, -6, 1, 4, 1])
print(r, "-> key không giảm?", check_order(r))

[5, -1, 4, 4, -6, -6, 1, 1, 1] -> key không giảm? True


Ở cell trên có **list comprehension** `[biểu_thức for x in result]`: tạo list mới bằng cách áp biểu thức cho từng phần tử. `all(...)` trả `True` nếu mọi điều kiện đều đúng.

In [23]:
print([x * x for x in [1, 2, 3]])
print(all([True, True]), all([True, False]))

[1, 4, 9]
True False


## 10. Độ phức tạp

Gọi `n = len(nums)`.

**Counter + sorted (lời giải chính):**

- Đếm: `O(n)` (tra cứu/cập nhật dict trung bình `O(1)`).
- Sắp xếp: `O(n log n)`; mỗi lần tính key là `O(1)` trung bình.
- Thời gian tổng: `O(n log n)`.
- Bộ nhớ: `O(n)` cho `freq` và list kết quả mới.

**Cách `nums.count`:** `O(n²)` thời gian. Với `n <= 100` vẫn pass, nhưng không nên tập thói quen này.

**Bucket:** `O(n)` để đếm và phân bucket, cộng sắp xếp các giá trị khác nhau `O(k log k)` với `k <= n`. Giá trị bị giới hạn `-100..100` nên `k <= 201`.

## 11. Bộ test

Nhóm test:

1. 3 ví dụ của đề.
2. Biên: 1 phần tử; tất cả giống nhau.
3. Tất cả khác nhau (cùng tần suất 1) → phải ra giảm dần.
4. Có số âm, có 0; giá trị biên `-100`, `100`.
5. Đối chiếu tất cả các cách trên nhiều input ngẫu nhiên (có seed).

In [24]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<28} PASS -> {actual}")

check("đề 1", frequency_sort_counter([1, 1, 2, 2, 2, 3]), [3, 1, 1, 2, 2, 2])
check("đề 2", frequency_sort_counter([2, 3, 1, 3, 2]), [1, 3, 3, 2, 2])
check("đề 3", frequency_sort_counter([-1, 1, -6, 4, 5, -6, 1, 4, 1]), [5, -1, 4, 4, -6, -6, 1, 1, 1])
check("một phần tử", frequency_sort_counter([7]), [7])
check("tất cả giống nhau", frequency_sort_counter([4, 4, 4]), [4, 4, 4])
check("tất cả khác nhau -> giảm", frequency_sort_counter([1, 5, -3, 0]), [5, 1, 0, -3])
check("biên -100 / 100", frequency_sort_counter([-100, 100, 100]), [-100, 100, 100])
check("hòa có số âm", frequency_sort_counter([-2, -2, -1, -1]), [-1, -1, -2, -2])

đề 1                         PASS -> [3, 1, 1, 2, 2, 2]
đề 2                         PASS -> [1, 3, 3, 2, 2]
đề 3                         PASS -> [5, -1, 4, 4, -6, -6, 1, 1, 1]
một phần tử                  PASS -> [7]
tất cả giống nhau            PASS -> [4, 4, 4]
tất cả khác nhau -> giảm     PASS -> [5, 1, 0, -3]
biên -100 / 100              PASS -> [-100, 100, 100]
hòa có số âm                 PASS -> [-1, -1, -2, -2]


In [25]:
import random

rng = random.Random(1636)
methods = [frequency_sort_count, frequency_sort_dict, frequency_sort_bucket,
           frequency_sort_cmp, frequency_sort_two_pass]
for _ in range(500):
    n = rng.randint(1, 30)
    arr = [rng.randint(-5, 5) for _ in range(n)]
    ref = frequency_sort_counter(arr)
    for m in methods:
        assert m(arr) == ref, (m.__name__, arr)
print("5 cách khác nhau cho cùng kết quả trên 500 input ngẫu nhiên")

5 cách khác nhau cho cùng kết quả trên 500 input ngẫu nhiên


## 12. Lỗi người mới hay gặp

### Lỗi 1: Sắp cả hai tiêu chí cùng chiều

Sai: `key=lambda x: (freq[x], x)` → hòa tần suất thì số **nhỏ** đứng trước.

In [26]:
from collections import Counter

def wrong_same_direction(nums):
    freq = Counter(nums)
    return sorted(nums, key=lambda x: (freq[x], x))

print("sai :", wrong_same_direction([2, 3, 1, 3, 2]))
print("đúng:", frequency_sort_counter([2, 3, 1, 3, 2]))

sai : [1, 2, 2, 3, 3]
đúng: [1, 3, 3, 2, 2]


### Lỗi 2: Dùng `reverse=True` cho toàn bộ

Sai: `sorted(nums, key=lambda x: (freq[x], x), reverse=True)` → đảo **cả hai** tiêu chí, tần suất thành giảm dần.

Sửa: chỉ đảo tiêu chí cần giảm bằng dấu trừ.

In [27]:
def wrong_reverse_all(nums):
    freq = Counter(nums)
    return sorted(nums, key=lambda x: (freq[x], x), reverse=True)

print("sai :", wrong_reverse_all([1, 1, 2, 2, 2, 3]))
print("đúng:", frequency_sort_counter([1, 1, 2, 2, 2, 3]))

sai : [2, 2, 2, 1, 1, 3]
đúng: [3, 1, 1, 2, 2, 2]


### Lỗi 3: Chỉ sắp các giá trị khác nhau rồi quên "bung" ra

Sai: `sorted(freq, key=...)` chỉ sắp **các key của dict** (mỗi giá trị một lần), kết quả bị thiếu phần tử.

Sửa: sắp `nums` (đủ phần tử), hoặc khi dùng bucket thì `extend([value] * f)`.

In [28]:
def wrong_unique_only(nums):
    freq = Counter(nums)
    return sorted(freq, key=lambda x: (freq[x], -x))

print("sai :", wrong_unique_only([1, 1, 2, 2, 2, 3]), "(thiếu phần tử)")
print("đúng:", frequency_sort_counter([1, 1, 2, 2, 2, 3]))

sai : [3, 1, 2] (thiếu phần tử)
đúng: [3, 1, 1, 2, 2, 2]


### Lỗi 4: `nums.sort()` rồi `return nums.sort()`

`list.sort()` trả về `None`. Nếu muốn sắp tại chỗ, viết `nums.sort(key=...)` rồi `return nums` ở dòng sau.

In [29]:
def wrong_return_sort(nums):
    freq = Counter(nums)
    return nums.sort(key=lambda x: (freq[x], -x))

def fixed_return_sort(nums):
    freq = Counter(nums)
    nums.sort(key=lambda x: (freq[x], -x))
    return nums

print("sai :", wrong_return_sort([2, 3, 1, 3, 2]))
print("đúng:", fixed_return_sort([2, 3, 1, 3, 2]))

sai : None
đúng: [1, 3, 3, 2, 2]


### Lỗi 5: Comparator trả về `True/False` thay vì số âm/dương

`cmp_to_key` cần số âm/0/dương. Trả về `freq[a] < freq[b]` (bool) sẽ cho `True = 1`, `False = 0`, không bao giờ âm → sắp sai. Luôn trả về hiệu `freq[a] - freq[b]` hoặc `-1/0/1`.

## 13. Mẹo nhận dạng pattern

- Đề nói "sort by frequency" / "theo số lần xuất hiện" → **đếm bằng dict/Counter trước**, rồi sort.
- "Nếu bằng nhau thì theo …" → **key là tuple** `(tiêu_chí_1, tiêu_chí_2, ...)`.
- Một tiêu chí tăng, một tiêu chí giảm và tiêu chí là **số** → **đảo dấu** tiêu chí giảm.
- Tiêu chí giảm là **chuỗi** (không đảo dấu được) → dùng hai lần stable sort hoặc `cmp_to_key`.
- Tần suất bị giới hạn bởi `n` → có thể nghĩ đến **bucket theo tần suất**.

Bài liên quan: 451 Sort Characters By Frequency, 347 Top K Frequent Elements, 692 Top K Frequent Words, 1122 Relative Sort Array.

## 14. Biến thể

### Biến thể 1: Tần suất giảm dần, hòa thì giá trị tăng dần

Chỉ cần đổi key thành `(-freq[x], x)`.

In [30]:
from collections import Counter

def frequency_sort_desc(nums):
    freq = Counter(nums)
    return sorted(nums, key=lambda x: (-freq[x], x))

print(frequency_sort_desc([2, 3, 1, 3, 2]))   # 2,2,3,3 rồi 1

[2, 2, 3, 3, 1]


### Biến thể 2: Sắp chuỗi theo tần suất ký tự (giống LeetCode 451)

Ký tự xuất hiện nhiều đứng trước; hòa thì theo thứ tự chữ cái. Ở đây phần tử là ký tự nên dùng `sorted(s, ...)` rồi `"".join(...)`.

In [31]:
def sort_chars_by_freq(s: str) -> str:
    freq = Counter(s)
    return "".join(sorted(s, key=lambda ch: (-freq[ch], ch)))

print(sort_chars_by_freq("tree"))
print(sort_chars_by_freq("cccaaa"))

eert
aaaccc


### Biến thể 3: Chỉ lấy danh sách giá trị khác nhau theo thứ tự đề

Lúc này mới đúng là sắp các **key của Counter**.

In [32]:
def distinct_by_frequency(nums):
    freq = Counter(nums)
    return sorted(freq, key=lambda x: (freq[x], -x))

print(distinct_by_frequency([-1, 1, -6, 4, 5, -6, 1, 4, 1]))

[5, -1, 4, -6, 1]


## 15. Bài tập tự luyện

Làm theo thứ tự. Các cell TODO chỉ định nghĩa hàm, **không** gọi khi chạy notebook. Bỏ dấu `#` ở các dòng test để tự kiểm tra.

### Bài 1 — Dự đoán output (tính tay)

Không chạy code, hãy viết ra key `(freq[x], -x)` của từng phần tử và kết quả của `frequency_sort_counter([3, 3, -2, 0, 0, 5])`.

### Bài 2 — Đếm tần suất không dùng Counter

In [33]:
def exercise_count(nums: list[int]) -> dict:
    # TODO: tạo dict rỗng, duyệt nums, tăng bộ đếm bằng .get(x, 0) + 1
    pass

# assert exercise_count([2, 3, 1, 3, 2]) == {2: 2, 3: 2, 1: 1}
# assert exercise_count([]) == {}

### Bài 3 — Tự viết lại lời giải chính

In [34]:
def exercise_frequency_sort(nums: list[int]) -> list[int]:
    # TODO 1: đếm tần suất
    # TODO 2: sorted với key là tuple (tiêu chí chính, tiêu chí phụ)
    pass

# assert exercise_frequency_sort([1, 1, 2, 2, 2, 3]) == [3, 1, 1, 2, 2, 2]
# assert exercise_frequency_sort([2, 3, 1, 3, 2]) == [1, 3, 3, 2, 2]

### Bài 4 — Viết cách bucket

Không gọi `sorted` trên toàn bộ `nums`; chỉ được sort trong từng bucket.

In [35]:
def exercise_bucket(nums: list[int]) -> list[int]:
    # TODO: buckets[f] = các giá trị có tần suất f; duyệt f tăng dần
    pass

# assert exercise_bucket([-1, 1, -6, 4, 5, -6, 1, 4, 1]) == [5, -1, 4, 4, -6, -6, 1, 1, 1]

### Bài 5 — Biến thể với chuỗi: sắp từ theo tần suất tăng, hòa thì theo thứ tự từ điển **giảm dần**

Ví dụ `["b", "a", "c", "a"]` → `"b"` và `"c"` 1 lần, hòa nên `"c"` trước `"b"`; rồi `"a","a"`.
Chuỗi không đảo dấu được — hãy dùng hai lần stable sort.

In [36]:
def exercise_words(words: list[str]) -> list[str]:
    # TODO: sort phụ (giảm dần theo chữ) trước, sort chính (tần suất tăng) sau
    pass

# assert exercise_words(["b", "a", "c", "a"]) == ["c", "b", "a", "a"]

### Bài 6 — Mini challenge: Top K phần tử ít xuất hiện nhất

Trả về `k` **giá trị khác nhau** có tần suất nhỏ nhất, hòa thì giá trị lớn hơn được ưu tiên.

In [37]:
def exercise_k_least_frequent(nums: list[int], k: int) -> list[int]:
    # TODO: đếm, sắp các giá trị khác nhau theo (freq, -x), lấy k phần tử đầu
    pass

# assert exercise_k_least_frequent([1, 1, 2, 2, 2, 3, 4], 2) == [4, 3]

## 16. Gợi ý 3 mức

**Bài 1**
- Mức 1: đếm trước: `3→2, -2→1, 0→2, 5→1`.
- Mức 2: các số tần suất 1 là `-2, 5`; tần suất 2 là `3, 0`.
- Mức 3: trong mỗi nhóm xếp giảm: `5, -2`, rồi `3, 3, 0, 0`.

**Bài 2**
- Mức 1: mỗi lần gặp `x` thì tăng bộ đếm của `x`.
- Mức 2: `d.get(x, 0)` trả 0 khi chưa có key.
- Mức 3: `d[x] = d.get(x, 0) + 1` trong vòng `for x in nums`.

**Bài 3**
- Mức 1: đếm một lần rồi tra cứu khi sort.
- Mức 2: key là tuple, tiêu chí giảm dùng dấu trừ.
- Mức 3: `sorted(nums, key=lambda x: (freq[x], -x))`.

**Bài 4**
- Mức 1: tần suất chỉ từ 1 đến `n`.
- Mức 2: `buckets = [[] for _ in range(n + 1)]`, dùng `freq.items()`.
- Mức 3: `for f in range(1, n+1): for v in sorted(buckets[f], reverse=True): res.extend([v]*f)`.

**Bài 5**
- Mức 1: stable sort giữ thứ tự khi key bằng nhau.
- Mức 2: lần 1 `sorted(words, reverse=True)`.
- Mức 3: lần 2 `sorted(step1, key=lambda w: freq[w])`.

**Bài 6**
- Mức 1: chỉ cần giá trị khác nhau → sort các key của Counter.
- Mức 2: key giống bài chính.
- Mức 3: `sorted(freq, key=lambda x: (freq[x], -x))[:k]`.

## 17. Đáp án tham khảo

Chỉ xem sau khi đã tự làm.

**Bài 1:** key: `3→(2,-3)`, `-2→(1,2)`, `0→(2,0)`, `5→(1,-5)`. Kết quả `[5, -2, 3, 3, 0, 0]`.

In [38]:
from collections import Counter

def answer_count(nums):
    d = {}
    for x in nums:
        d[x] = d.get(x, 0) + 1
    return d

def answer_frequency_sort(nums):
    freq = Counter(nums)
    return sorted(nums, key=lambda x: (freq[x], -x))

def answer_bucket(nums):
    freq = Counter(nums)
    n = len(nums)
    buckets = [[] for _ in range(n + 1)]
    for v, f in freq.items():
        buckets[f].append(v)
    res = []
    for f in range(1, n + 1):
        for v in sorted(buckets[f], reverse=True):
            res.extend([v] * f)
    return res

def answer_words(words):
    freq = Counter(words)
    step1 = sorted(words, reverse=True)
    return sorted(step1, key=lambda w: freq[w])

def answer_k_least_frequent(nums, k):
    freq = Counter(nums)
    return sorted(freq, key=lambda x: (freq[x], -x))[:k]

assert answer_frequency_sort([3, 3, -2, 0, 0, 5]) == [5, -2, 3, 3, 0, 0]   # Bài 1
assert answer_count([2, 3, 1, 3, 2]) == {2: 2, 3: 2, 1: 1}
assert answer_count([]) == {}
assert answer_frequency_sort([1, 1, 2, 2, 2, 3]) == [3, 1, 1, 2, 2, 2]
assert answer_frequency_sort([2, 3, 1, 3, 2]) == [1, 3, 3, 2, 2]
assert answer_bucket([-1, 1, -6, 4, 5, -6, 1, 4, 1]) == [5, -1, 4, 4, -6, -6, 1, 1, 1]
assert answer_words(["b", "a", "c", "a"]) == ["c", "b", "a", "a"]
assert answer_k_least_frequent([1, 1, 2, 2, 2, 3, 4], 2) == [4, 3]
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 18. Template nhớ nhanh + checklist submit

```python
from collections import Counter

class Solution:
    def frequencySort(self, nums: list[int]) -> list[int]:
        freq = Counter(nums)                              # 1) đếm một lần
        return sorted(nums, key=lambda x: (freq[x], -x))  # 2) tuple key: tăng freq, giảm giá trị
```

Công thức tổng quát: **"sắp theo A tăng, hòa thì B giảm" → `key=lambda x: (A(x), -B(x))`**.

Checklist trước khi submit:

- [ ] Đã đếm tần suất **trước** khi sort (không gọi `nums.count` trong key)?
- [ ] Tiêu chí chính là tần suất **tăng**, tiêu chí phụ là giá trị **giảm** (có dấu `-`)?
- [ ] Không dùng `reverse=True` cho toàn bộ?
- [ ] Sort trên `nums` (đủ phần tử), không phải trên các key của Counter?
- [ ] Trả về list, không trả về kết quả của `.sort()` (là `None`)?
- [ ] Đã thử ví dụ có số âm như `[-1,1,-6,4,5,-6,1,4,1]`?